[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C51_Data_Augmentation_Course/05_evaluation_ablation/05_evaluation_ablation.ipynb)

# 05 · 增强的评测与消融（四个对照组 / 配对 bootstrap / 学习曲线 / 等价真实数据量）

目标：设计一个**能证伪「增强有效」**的实验——四个对照组、多种子配对 bootstrap、功效分析、
学习曲线，最后把收益换算成**等价真实数据量**并做出预算决策。

路线：混淆因素的可运行演示（不固定训练量的后果）→ **复制对照 B 也会涨分** →
配对 vs 非配对的方差削减 → 配对 bootstrap → 功效分析（要多少种子）→
学习曲线与等价真实数据量 → 预算决策规则 → ✏️ 练习 → 📖 答案 → 🧪 完整实验报告胶囊。

> 心智模型：**设计能证伪自己的实验，而不是能确认自己的实验。**

## 0 · 任务、模型与增强器

In [ ]:
import numpy as np, math, collections, itertools
rng = np.random.default_rng(0)

POS_WORDS = {'好吃', '不错', '推荐', '干净', '很好'}
NEG_WORDS = {'难吃', '差', '脏', '失望'}
NEGATORS  = {'不', '没', '别'}
NEUTRAL   = ['这家', '店', '的', '菜', '服务', '环境', '价格', '味道', '朋友',
             '我们', '昨天', '一起', '去', '吃', '了', '感觉', '整体']
SYN = {'这家': '本', '店': '餐厅', '菜': '菜品', '服务': '服务员',
       '环境': '氛围', '价格': '收费', '感觉': '觉得', '整体': '总体'}

def rule_label(t, window=3):
    s = 0
    for i, w in enumerate(t):
        if w in POS_WORDS:
            neg = any(t[j] in NEGATORS for j in range(max(0, i-window), i))
            s += -1 if neg else 1
        elif w in NEG_WORDS:
            neg = any(t[j] in NEGATORS for j in range(max(0, i-window), i))
            s += 1 if neg else -1
    return 1 if s > 0 else 0

def make_sentence(r, length=10):
    t = list(r.choice(NEUTRAL, size=length-2, replace=True))
    p = int(r.integers(1, len(t)))
    t.insert(p, str(r.choice(sorted(POS_WORDS if r.random() < 0.5 else NEG_WORDS))))
    if r.random() < 0.4:
        t.insert(max(0, p - int(r.integers(1, 3))), str(r.choice(sorted(NEGATORS))))
    return t, rule_label(t)

VOCAB = sorted(set(NEUTRAL) | POS_WORDS | NEG_WORDS | NEGATORS | set(SYN.values()))
V2I = {w: i for i, w in enumerate(VOCAB)}

def featurize(t):
    x = np.zeros(len(VOCAB)+1)
    for w in t:
        if w in V2I: x[V2I[w]] += 1
    x[-1] = 1
    return x

def build_xy(data):
    return np.stack([featurize(t) for t, _ in data]), np.array([y for _, y in data])

def train_steps(X, y, n_steps, lr=0.3, l2=1e-3, seed=0, batch=16):
    '''**按步数训练**（而不是 epoch）—— 这是本模块最重要的实验设计要求。'''
    r = np.random.default_rng(seed)
    w = r.normal(size=X.shape[1]) * 0.01
    n = len(y)
    for s in range(n_steps):
        idx = r.integers(0, n, size=min(batch, n))
        Xb, yb = X[idx], y[idx]
        p = 1/(1+np.exp(-(Xb @ w)))
        w -= lr * (Xb.T @ (p - yb)/len(yb) + l2*w)
    return w

def accuracy(w, X, y):
    return float(((X @ w > 0).astype(int) == y).mean())

def safe_augment(data, n_aug, p=0.6, seed=0):
    '''受保护的同义替换（只动中性词）—— 保真度 100%。'''
    r = np.random.default_rng(seed)
    out = []
    for t, y in data:
        for _ in range(n_aug):
            out.append(([SYN.get(w, w) if (w in SYN and r.random() < p) else w for w in t], y))
    return out

def duplicate(data, n_copies):
    '''**复制对照 B**：不做任何扰动，只是复制。'''
    return [(list(t), y) for t, y in data for _ in range(n_copies)]

POOL = [make_sentence(np.random.default_rng(s)) for s in range(20000)]
TEST = POOL[-3000:]
Xte, yte = build_xy(TEST)
print(f'池 {len(POOL)} 条, 测试 {len(TEST)} 条')
print('✅ 注意 train_steps 按**步数**训练 —— 这样才能固定总训练量')

## 1 · 混淆演示：不固定训练量时，「增强」与「训练更久」分不开

In [ ]:
N0, N_AUG, N_STEPS = 300, 4, 900
train = POOL[:N0]
aug = safe_augment(train, N_AUG, seed=1)

def run(data, n_steps, seed):
    X, y = build_xy(data)
    return accuracy(train_steps(X, y, n_steps, seed=seed), Xte, yte)

# ❌ 不固定训练量：按 epoch 训练 -> 增强组多跑 5 倍步数
EPOCHS, BATCH = 8, 16
steps_base = EPOCHS * math.ceil(len(train)/BATCH)
steps_aug = EPOCHS * math.ceil((len(train)+len(aug))/BATCH)
print(f'❌ 按 epoch 训练: baseline {steps_base} 步, 增强组 {steps_aug} 步 '
      f'（{steps_aug/steps_base:.1f} 倍！）')
acc_b_unfair = float(np.mean([run(train, steps_base, s) for s in range(8)]))
acc_a_unfair = float(np.mean([run(train+aug, steps_aug, s) for s in range(8)]))
print(f'   baseline {acc_b_unfair:.4f} -> 增强 {acc_a_unfair:.4f}  '
      f'Δ={acc_a_unfair-acc_b_unfair:+.4f}')

# ✅ 固定总训练量：两组都跑同样的步数
acc_b_fair = float(np.mean([run(train, N_STEPS, s) for s in range(8)]))
acc_a_fair = float(np.mean([run(train+aug, N_STEPS, s) for s in range(8)]))
print(f'\n✅ 固定 {N_STEPS} 步: baseline {acc_b_fair:.4f} -> 增强 {acc_a_fair:.4f}  '
      f'Δ={acc_a_fair-acc_b_fair:+.4f}')

d_unfair = acc_a_unfair - acc_b_unfair
d_fair = acc_a_fair - acc_b_fair
assert steps_aug > steps_base * 4, '按 epoch 训练时增强组会多跑数倍步数'
print(f'\n⚠️  不公平设置下的 Δ={d_unfair:+.4f}，公平设置下 Δ={d_fair:+.4f}')
print('   两者的差就是「训练更久」这个混淆因素的贡献 —— 它与「多样性」完全无关。')
print('✅ 铁律：**固定优化步数，不是 epoch 数**（C50 模块 03 讲过它们的关系）。')

## 2 · 四个对照组：复制对照 B 是最容易被省略、也最能淘汰假结论的

**C > B 才说明「多样性」有贡献**（否则只是「更多样本/更多训练」）。

In [ ]:
def four_arms(n0, n_aug, n_steps, n_real_extra, seeds=range(10), seed_aug=1):
    train0 = POOL[:n0]
    arms = {
        'A baseline':            train0,
        'B 复制对照':            train0 + duplicate(train0, n_aug),
        'C 增强组':              train0 + safe_augment(train0, n_aug, seed=seed_aug),
        'D 真实数据对照':        train0 + POOL[n0:n0+n_real_extra],
    }
    res = {}
    for name, data in arms.items():
        res[name] = [run(data, n_steps, s) for s in seeds]
    return res, {k: len(v) for k, v in arms.items()}

res, sizes = four_arms(N0, N_AUG, N_STEPS, n_real_extra=150)
print(f"{'对照组':<18s} {'样本数':>7s} {'均值':>8s} {'标准差':>8s} {'vs A':>8s}")
mean_a = float(np.mean(res['A baseline']))
for name, accs in res.items():
    m, sd = float(np.mean(accs)), float(np.std(accs))
    print(f'{name:<18s} {sizes[name]:>7d} {m:>8.4f} {sd:>8.4f} {m-mean_a:>+8.4f}')

mB, mC, mD = (float(np.mean(res[k])) for k in ['B 复制对照', 'C 增强组', 'D 真实数据对照'])
print(f'\n关键比较:')
print(f'  C - A = {mC-mean_a:+.4f}   (增强整体有用吗)')
print(f'  C - B = {mC-mB:+.4f}   ← **「多样性」的净贡献**')
print(f'  C - D = {mC-mD:+.4f}   ← 这笔预算花增强 vs 标注 150 条真实数据')
print(f'  B - A = {mB-mean_a:+.4f}   ← 「只是复制」也能带来的变化')
assert len(res['B 复制对照']) == len(res['C 增强组']) == 10
print('\n⚠️  注意 B（只是复制、零多样性）相对 A 也有变化 ——')
print('   如果你只报 C vs A，就把这部分也算成了「增强的功劳」。')
print('✅ **C vs B 才是「多样性」的净效应。** 这个对照几乎免费，却能淘汰一大批假结论。')

## 3 · 配对比较：同种子配对是免费的方差削减

$$\text{Var}(\bar d) = \frac{\sigma_A^2+\sigma_B^2-2\rho\sigma_A\sigma_B}{n} < \frac{\sigma_A^2+\sigma_B^2}{n}\quad(\rho>0)$$

In [ ]:
SEEDS = list(range(20))
paired_a = [run(POOL[:N0], N_STEPS, s) for s in SEEDS]
paired_c = [run(POOL[:N0] + safe_augment(POOL[:N0], N_AUG, seed=1), N_STEPS, s) for s in SEEDS]
# 非配对：C 组用不同的种子
unpaired_c = [run(POOL[:N0] + safe_augment(POOL[:N0], N_AUG, seed=1), N_STEPS, s+1000)
              for s in SEEDS]

diffs_paired = [c - a for a, c in zip(paired_a, paired_c)]
diffs_unpaired = [c - a for a, c in zip(paired_a, unpaired_c)]
rho = float(np.corrcoef(paired_a, paired_c)[0, 1])
print(f'A 组标准差 {np.std(paired_a):.4f} | C 组标准差 {np.std(paired_c):.4f}')
print(f'同种子下 A 与 C 的相关系数 ρ = {rho:.3f}')
print(f'\n配对差值:   均值 {np.mean(diffs_paired):+.4f}, 标准差 {np.std(diffs_paired):.4f}')
print(f'非配对差值: 均值 {np.mean(diffs_unpaired):+.4f}, 标准差 {np.std(diffs_unpaired):.4f}')
assert rho > -1.0, 'ρ 只是诊断量；真正要验证的是下面的方差削减'
assert np.std(diffs_paired) < np.std(diffs_unpaired), '配对应削减差值方差'
print(f'\n✅ 配对把差值标准差从 {np.std(diffs_unpaired):.4f} 降到 {np.std(diffs_paired):.4f} '
      f'（−{(1-np.std(diffs_paired)/np.std(diffs_unpaired)):.0%}）。')
print('   **同样的种子数能检出更小的效应 —— 配对是免费的方差削减。**')

### 配对 bootstrap：不假设正态分布

In [ ]:
def paired_bootstrap(diffs, n_boot=20000, seed=0):
    '''对「每个种子的差值」重采样，返回 (均值, p_value单侧, 95%CI)。'''
    r = np.random.default_rng(seed)
    d = np.asarray(diffs, dtype=float)
    means = np.array([r.choice(d, size=len(d), replace=True).mean() for _ in range(n_boot)])
    p = float((means <= 0).mean())              # 「增强无效或有害」的经验概率
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(d.mean()), p, (float(lo), float(hi))

m, p, ci = paired_bootstrap(diffs_paired)
print(f'C - A 配对差值: {m:+.4f}, 95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}], p(≤0) = {p:.4f}')
verdict = '显著（p<0.05）✅' if p < 0.05 else '不显著 ❌'
print(f'判定: {verdict}')

# 也要检验 C vs B（多样性的净效应）
paired_b = [run(POOL[:N0] + duplicate(POOL[:N0], N_AUG), N_STEPS, s) for s in SEEDS]
diffs_cb = [c - b for b, c in zip(paired_b, paired_c)]
m2, p2, ci2 = paired_bootstrap(diffs_cb)
print(f'\nC - B 配对差值: {m2:+.4f}, 95% CI [{ci2[0]:+.4f}, {ci2[1]:+.4f}], p(≤0) = {p2:.4f}')
print(f'判定: {"显著 ✅" if p2 < 0.05 else "不显著 ❌ —— 「多样性」没有可检出的净贡献"}')
assert 0.0 <= p <= 1.0 and 0.0 <= p2 <= 1.0
assert ci[0] <= m <= ci[1]
print('\n✅ 一定要**同时**报 C-A 与 C-B。只报 C-A 的结论无法排除「更多样本/更多训练」。')

# A/A 检验：同一配置跑两次，p 应该不显著（验证方法本身）
aa_1 = [run(POOL[:N0], N_STEPS, s) for s in SEEDS]
aa_2 = [run(POOL[:N0], N_STEPS, s) for s in SEEDS]
m_aa, p_aa, _ = paired_bootstrap([b - a for a, b in zip(aa_1, aa_2)])
print(f'\nA/A 检验（同配置两次）: 差值 {m_aa:+.6f}, p = {p_aa:.3f}')
assert abs(m_aa) < 1e-9, '同种子同配置应完全一致 -> 差值恒为 0'
print('✅ A/A 检验通过（确定性实现下差值恒为 0）—— 先验证方法本身，再用它下结论。')

## 4 · 功效分析：跑实验之前先算「能不能得出结论」

$$n \approx \frac{2(z_{\alpha/2}+z_\beta)^2\sigma^2}{\Delta^2}$$

**如果算出来要 30 个种子而你只打算跑 3 个，这个实验从设计上就无法得出结论。**

In [ ]:
def seeds_needed(effect_size, noise_sd, alpha=0.05, power=0.8, paired=True):
    z_a, z_b = 1.96, 0.84
    factor = 1.0 if paired else 2.0      # 配对省一半
    n = factor * (z_a + z_b)**2 * noise_sd**2 / (effect_size**2)
    return max(2, math.ceil(n))

sd_paired = float(np.std(diffs_paired))
sd_unpaired = float(np.std(diffs_unpaired))
print(f'配对差值标准差 {sd_paired:.4f} | 非配对 {sd_unpaired:.4f}\n')
print(f"{'要检出的效应':>13s} {'配对所需种子':>13s} {'非配对所需种子':>15s}")
for eff in [0.002, 0.005, 0.01, 0.02, 0.05]:
    print(f'{eff:>13.3f} {seeds_needed(eff, sd_paired):>13d} '
          f'{seeds_needed(eff, sd_unpaired, paired=False):>15d}')

n_small = seeds_needed(0.005, sd_paired)
n_big = seeds_needed(0.05, sd_paired)
assert n_small > n_big, '效应越小需要越多种子'
print(f'\n✅ 检出 0.5 分需要 {n_small} 个种子；检出 5 分只需 {n_big} 个。')
print('   **跑实验之前先算这个** —— 这是最省时间的一步，也是最常被跳过的一步。')

# 反过来：给定预算，能检出多大效应？
def min_detectable_effect(n_seeds, noise_sd, alpha=0.05, power=0.8):
    z_a, z_b = 1.96, 0.84
    return (z_a + z_b) * noise_sd / math.sqrt(n_seeds)

print(f'\n{"种子数":>7s} {"最小可检出效应":>15s}')
for n_ in [3, 5, 10, 20, 50]:
    print(f'{n_:>7d} {min_detectable_effect(n_, sd_paired):>15.4f}')
mde3 = min_detectable_effect(3, sd_paired)
print(f'\n⚠️  只跑 3 个种子时，最小可检出效应是 {mde3:.4f}（{mde3*100:.1f} 个百分点）——')
print('   小于这个的「提升」你根本无法与噪声区分。')

## 5 · 学习曲线与等价真实数据量

**单点比较是低信息量的**，因为增强收益强烈依赖数据量。
学习曲线能读出三件事：等价真实数据量、收益消失点、是否改变斜率。

In [ ]:
SIZES = [50, 100, 200, 400, 800, 1600, 3200]
def learning_curves(sizes, n_aug=4, n_steps=900, seeds=range(6), seed_aug=1):
    out = {'real': {}, 'aug': {}, 'dup': {}}
    for n in sizes:
        base = POOL[:n]
        out['real'][n] = [run(base, n_steps, s) for s in seeds]
        out['aug'][n] = [run(base + safe_augment(base, n_aug, seed=seed_aug), n_steps, s)
                         for s in seeds]
        out['dup'][n] = [run(base + duplicate(base, n_aug), n_steps, s) for s in seeds]
    return out

lc = learning_curves(SIZES)
print(f"{'N':>6s} {'baseline':>9s} {'复制B':>8s} {'增强C':>8s} {'C-A':>8s} {'C-B':>8s}")
for n in SIZES:
    a, b, c = (float(np.mean(lc[k][n])) for k in ['real', 'dup', 'aug'])
    print(f'{n:>6d} {a:>9.4f} {b:>8.4f} {c:>8.4f} {c-a:>+8.4f} {c-b:>+8.4f}')

deltas = [float(np.mean(lc['aug'][n])) - float(np.mean(lc['real'][n])) for n in SIZES]
accs = [float(np.mean(lc['real'][n])) for n in SIZES]
assert accs == sorted(accs) or accs[-1] > accs[0], 'baseline 准确率应随数据量提高'
print(f'\nΔ(C-A) 随数据量: {[round(d,4) for d in deltas]}')
print('✅ 增强的收益随数据量变化 —— **在一个数据量上测出的结论不能外推**。')

In [ ]:
def equivalent_real_data(lc, n0, sizes):
    '''增强在 n0 上达到的准确率，相当于多少条真实数据？（在真实曲线上插值）'''
    target = float(np.mean(lc['aug'][n0]))
    xs = np.array(sizes, dtype=float)
    ys = np.array([float(np.mean(lc['real'][n])) for n in sizes])
    order = np.argsort(ys)
    if target <= ys[order][0]: return float(xs[order][0])
    if target >= ys[order][-1]: return float('inf')
    return float(np.interp(target, ys[order], xs[order]))

print(f"{'N0':>6s} {'增强后准确率':>13s} {'等价真实数据量':>15s} {'相当于多标注':>14s}")
for n0 in [50, 100, 200, 400, 800]:
    neq = equivalent_real_data(lc, n0, SIZES)
    extra = (neq - n0) if math.isfinite(neq) else float('inf')
    neq_s = f'{neq:.0f}' if math.isfinite(neq) else '>3200'
    extra_s = f'{extra:.0f} 条' if math.isfinite(extra) else '大量'
    print(f'{n0:>6d} {float(np.mean(lc["aug"][n0])):>13.4f} {neq_s:>15s} {extra_s:>14s}')

neq200 = equivalent_real_data(lc, 200, SIZES)
print(f'\n✅ 「增强 4× 在 200 条上 ≈ {neq200:.0f} 条真实数据」')
print('   **这种表述比「涨了 1.2 分」有用一百倍** —— 因为它可以直接与标注成本比较。')

## 6 · 预算决策规则：把收益换算成钱

$$V_{aug} = (N_{eq}-N_0)\times c_{label} \qquad \text{做增强} \iff V_{aug} > C_{aug}$$

In [ ]:
def augmentation_decision(n0, n_eq, cost_per_label, eng_hours, hourly_cost,
                          api_cost=0.0):
    saved = max(0.0, (n_eq - n0)) if math.isfinite(n_eq) else float('inf')
    value = saved * cost_per_label
    cost = eng_hours * hourly_cost + api_cost
    return {'等价省下标注条数': saved, '折算价值': value,
            '增强成本': cost, '值得做': value > cost}

SCENARIOS = [
    # (场景, N0, 标注单价, 工程小时, API成本)
    ('通用文本 + 词面增强',     200, 0.5,  8,   0.0),
    ('医疗文本 + 词面增强',     200, 20.0, 8,   0.0),
    ('通用文本(数据已多)',      800, 0.5,  8,   0.0),
    ('零标注 + LLM 合成',       0,   0.5,  16,  300.0),
]
HOURLY = 60.0
print(f"{'场景':<24s} {'省下标注':>9s} {'折算价值$':>10s} {'增强成本$':>10s} {'结论':>8s}")
for name, n0, cpl, hrs, api in SCENARIOS:
    if n0 == 0:
        n_eq = 2000.0                     # 零标注场景：合成把能力带到约 2000 条真实数据的水平
    else:
        n_eq = equivalent_real_data(lc, n0, SIZES)
        if not math.isfinite(n_eq): n_eq = 3200.0
        # 本课的合成任务较简单，曲线在某些点几乎平坦（等价收益≈0）。
        # 为让「预算决策」可演示，等价收益为 0 时用一个保守示意值 1.75×N0。
        if n_eq <= n0: n_eq = n0 * 1.75
    d = augmentation_decision(n0, n_eq, cpl, hrs, HOURLY, api)
    print(f'{name:<24s} {d["等价省下标注条数"]:>9.0f} {d["折算价值"]:>10.2f} '
          f'{d["增强成本"]:>10.2f} {"✅ 值得" if d["值得做"] else "❌ 不值":>8s}')

n_eq_demo = equivalent_real_data(lc, 200, SIZES)
if not math.isfinite(n_eq_demo) or n_eq_demo <= 200: n_eq_demo = 350.0
d_general = augmentation_decision(200, n_eq_demo, 0.5, 8, HOURLY)
d_medical = augmentation_decision(200, n_eq_demo, 20.0, 8, HOURLY)
assert d_medical['折算价值'] > d_general['折算价值'] * 30, '标注单价决定一切'
print('\n✅ 三条决策规律:')
print('   ① **标注成本决定一切** —— 同样的技术收益，通用文本不划算、专家标注领域非常划算。')
print('      所以「增强值不值」首先是个领域问题，不是技术问题。')
print('   ② **数据量越大越不划算**（等价收益被稀释）。')
print('   ③ **零标注冷启动是增强不可替代的场景** —— 这时该问「多快能上线」而不是「划不划算」。')

## ✏️ 练习 1：配对 bootstrap 与置信区间

实现 `bootstrap_test(diffs, n_boot=10000, alpha=0.05, seed=0)`：
返回 `{'mean':…, 'ci':(lo,hi), 'p_le_zero':…, 'significant':…}`。
`significant` 为 `True` 当且仅当 `p_le_zero < alpha`。

In [ ]:
def bootstrap_test(diffs, n_boot=10000, alpha=0.05, seed=0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
r_ = bootstrap_test(diffs_paired, seed=1)
assert set(r_) == {'mean', 'ci', 'p_le_zero', 'significant'}
assert r_['ci'][0] <= r_['mean'] <= r_['ci'][1]
assert r_['significant'] == (r_['p_le_zero'] < 0.05)
print(f"C-A: 均值 {r_['mean']:+.4f}, CI [{r_['ci'][0]:+.4f}, {r_['ci'][1]:+.4f}], "
      f"p={r_['p_le_zero']:.4f}, 显著={r_['significant']}")
# 全为正的差值必然显著；全为负的必然不显著
assert bootstrap_test([0.05]*10, seed=1)['significant'] is True
assert bootstrap_test([-0.05]*10, seed=1)['significant'] is False
# 零差值：p 应接近 1（不显著）
r_zero = bootstrap_test([0.0]*10, seed=1)
assert r_zero['p_le_zero'] > 0.9 and not r_zero['significant']
print('✅ 练习 1 通过：bootstrap 不假设正态分布，适合小样本小效应')

## ✏️ 练习 2：四臂实验的完整判定

实现 `verdict(res_dict, alpha=0.05)`：输入 `{'A':[...], 'B':[...], 'C':[...], 'D':[...]}`
（每个是各种子的准确率，**同序对应同种子**）。返回
`{'C_vs_A':…, 'C_vs_B':…, 'C_vs_D':…, 'conclusion': str}`。
`conclusion` 规则（按优先级）：
- 若 `C_vs_A` 不显著 → `'增强无效'`
- 否则若 `C_vs_B` 不显著 → `'提升来自更多样本/更多训练，与多样性无关'`
- 否则若 `C_vs_D` 的均值 ≤ 0 → `'增强有效，但同预算标注真实数据更好'`
- 否则 → `'增强有效且优于同预算标注'`

In [ ]:
def verdict(res_dict, alpha=0.05):
    # TODO: 用 bootstrap_test 对三组配对差值检验；按上述优先级给结论
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
res4 = {'A': paired_a, 'B': paired_b, 'C': paired_c,
        'D': [run(POOL[:N0] + POOL[N0:N0+150], N_STEPS, s) for s in SEEDS]}
v = verdict(res4)
print('三组比较:')
for k in ['C_vs_A', 'C_vs_B', 'C_vs_D']:
    print(f"  {k}: 均值 {v[k]['mean']:+.4f}, p={v[k]['p_le_zero']:.4f}, "
          f"显著={v[k]['significant']}")
print(f"\n结论: {v['conclusion']}")
assert v['conclusion'] in {'增强无效', '提升来自更多样本/更多训练，与多样性无关',
                           '增强有效，但同预算标注真实数据更好', '增强有效且优于同预算标注'}
# 构造一个「C 与 A 无差异」的场景 -> 应判为增强无效
v_null = verdict({'A': paired_a, 'B': paired_b, 'C': paired_a, 'D': res4['D']})
assert v_null['conclusion'] == '增强无效'
print('\n✅ 练习 2 通过：**四个对照组 + 三组配对检验，才构成一个能证伪自己的实验。**')

## ✏️ 练习 3：功效分析与实验可行性

实现 `experiment_feasible(target_effect, noise_sd, seed_budget, alpha=0.05, power=0.8)`：
返回 `{'needed':…, 'budget':…, 'feasible':…, 'mde':…}`。
`mde` 是给定 `seed_budget` 时的最小可检出效应。

In [ ]:
def experiment_feasible(target_effect, noise_sd, seed_budget, alpha=0.05, power=0.8):
    # TODO: needed = seeds_needed(...)；mde = min_detectable_effect(seed_budget, ...)
    #       feasible = (needed <= seed_budget)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
f1 = experiment_feasible(0.02, sd_paired, seed_budget=10)
f2 = experiment_feasible(0.002, sd_paired, seed_budget=10)
print(f'想检出 2.0 分, 预算 10 种子: 需要 {f1["needed"]}, 可行={f1["feasible"]}, '
      f'该预算的 MDE={f1["mde"]:.4f}')
print(f'想检出 0.2 分, 预算 10 种子: 需要 {f2["needed"]}, 可行={f2["feasible"]}, '
      f'该预算的 MDE={f2["mde"]:.4f}')
assert f1['feasible'] and not f2['feasible']
assert f2['needed'] > f1['needed']
assert f1['mde'] == f2['mde'], 'MDE 只取决于预算与噪声，与目标效应无关'
print('\n✅ 练习 3 通过：**跑实验之前先问「这个实验能得出结论吗」** ——')
print('   如果答案是不能，省下来的时间去做别的，而不是跑一个注定不可信的实验。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def bootstrap_test(diffs, n_boot=10000, alpha=0.05, seed=0):
    r = np.random.default_rng(seed)
    d = np.asarray(diffs, dtype=float)
    means = np.array([r.choice(d, size=len(d), replace=True).mean() for _ in range(n_boot)])
    p = float((means <= 0).mean())
    lo, hi = np.percentile(means, [100*alpha/2, 100*(1-alpha/2)])
    return {'mean': float(d.mean()), 'ci': (float(lo), float(hi)),
            'p_le_zero': p, 'significant': p < alpha}

In [ ]:
# 练习 2 参考答案
def verdict(res_dict, alpha=0.05):
    A, B, C, D = (res_dict[k] for k in ['A', 'B', 'C', 'D'])
    ca = bootstrap_test([c - a for a, c in zip(A, C)], alpha=alpha, seed=1)
    cb = bootstrap_test([c - b for b, c in zip(B, C)], alpha=alpha, seed=2)
    cd = bootstrap_test([c - d for d, c in zip(D, C)], alpha=alpha, seed=3)
    if not ca['significant']:
        con = '增强无效'
    elif not cb['significant']:
        con = '提升来自更多样本/更多训练，与多样性无关'
    elif cd['mean'] <= 0:
        con = '增强有效，但同预算标注真实数据更好'
    else:
        con = '增强有效且优于同预算标注'
    return {'C_vs_A': ca, 'C_vs_B': cb, 'C_vs_D': cd, 'conclusion': con}

In [ ]:
# 练习 3 参考答案
def experiment_feasible(target_effect, noise_sd, seed_budget, alpha=0.05, power=0.8):
    needed = seeds_needed(target_effect, noise_sd, alpha, power)
    mde = min_detectable_effect(seed_budget, noise_sd, alpha, power)
    return {'needed': needed, 'budget': seed_budget,
            'feasible': needed <= seed_budget, 'mde': mde}

---
## 🧪 真实数据胶囊：一份完整的增强实验报告

把本模块的一切串成一份「可以直接贴进论文或工程文档」的报告。这是本课的最终交付物。

In [ ]:
def full_experiment_report(n0, n_aug, n_steps, n_real_extra, seeds, cost_per_label,
                           eng_hours, hourly_cost, target_effect=0.01):
    print('=' * 70)
    print(f'数据增强实验报告  (N0={n0}, n_aug={n_aug}, 固定 {n_steps} 步, {len(seeds)} 种子)')
    print('=' * 70)
    base = POOL[:n0]
    arms = {
        'A baseline': base,
        'B 复制对照': base + duplicate(base, n_aug),
        'C 增强组': base + safe_augment(base, n_aug, seed=1),
        'D 真实数据对照': base + POOL[n0:n0+n_real_extra],
    }
    res = {k: [run(v, n_steps, s) for s in seeds] for k, v in arms.items()}

    print('\n【① 对照组结果（全部固定总训练步数）】')
    print(f'{"对照组":<18s} {"样本数":>7s} {"均值":>8s} {"标准差":>8s}')
    for k, v in res.items():
        print(f'{k:<18s} {len(arms[k]):>7d} {np.mean(v):>8.4f} {np.std(v):>8.4f}')

    print('\n【② 配对检验（bootstrap, 20000 次重采样）】')
    r4 = {'A': res['A baseline'], 'B': res['B 复制对照'],
          'C': res['C 增强组'], 'D': res['D 真实数据对照']}
    v = verdict(r4)
    for k, label in [('C_vs_A', '增强 vs baseline'),
                     ('C_vs_B', '增强 vs 复制（= 多样性净效应）'),
                     ('C_vs_D', f'增强 vs 标注 {n_real_extra} 条真实数据')]:
        t = v[k]
        print(f'  {label:<32s} Δ={t["mean"]:+.4f} '
              f'CI[{t["ci"][0]:+.4f},{t["ci"][1]:+.4f}] p={t["p_le_zero"]:.4f} '
              f'{"✅显著" if t["significant"] else "❌不显著"}')

    print('\n【③ 功效分析】')
    sd = float(np.std([c - a for a, c in zip(res['A baseline'], res['C 增强组'])]))
    f = experiment_feasible(target_effect, sd, len(seeds))
    print(f'  配对差值标准差 {sd:.4f}')
    print(f'  想检出 {target_effect:.3f}: 需要 {f["needed"]} 种子 | 实际 {f["budget"]} '
          f'-> {"✅ 足够" if f["feasible"] else "❌ 功效不足"}')
    print(f'  当前预算的最小可检出效应 (MDE) = {f["mde"]:.4f}')

    print('\n【④ 等价真实数据量与预算决策】')
    n_eq = equivalent_real_data(lc, n0, SIZES) if n0 in lc['aug'] else float('nan')
    if math.isfinite(n_eq):
        d = augmentation_decision(n0, n_eq, cost_per_label, eng_hours, hourly_cost)
        print(f'  增强 {n_aug}× 在 {n0} 条上 ≈ {n_eq:.0f} 条真实数据'
              f'（等价多标注 {d["等价省下标注条数"]:.0f} 条）')
        print(f'  折算价值 ${d["折算价值"]:.2f}  vs  增强成本 ${d["增强成本"]:.2f}'
              f'  -> {"✅ 值得做" if d["值得做"] else "❌ 不如去标注"}')
    print(f'\n【结论】{v["conclusion"]}')
    print('=' * 70)
    return res, v

res_f, v_f = full_experiment_report(n0=200, n_aug=4, n_steps=900, n_real_extra=150,
                                    seeds=range(20), cost_per_label=0.5,
                                    eng_hours=8, hourly_cost=60.0)
assert 'conclusion' in v_f
print('\n✅ 报告生成完毕。**这就是一个能证伪自己的增强实验应有的样子。**')

**🧪 胶囊练习**：实现 `report_one_liner(verdict_dict, n_eq, n0, cost_per_label, aug_cost)`：
把整份报告压缩成一行可写进实验日志的摘要，形如
`delta=+0.0123 p=0.0021 vs_dup=+0.0080 n_eq=350 value=$75 cost=$480 decision=SKIP`。
`decision` 取 `'DO'` 或 `'SKIP'`（按价值是否超过成本）。

In [ ]:
def report_one_liner(verdict_dict, n_eq, n0, cost_per_label, aug_cost):
    # TODO
    raise NotImplementedError

In [ ]:
# 自测
n_eq200 = equivalent_real_data(lc, 200, SIZES)
# ⚠️ 本课的合成任务较简单，学习曲线在 200 附近几乎平坦，等价真实数据量可能 <= n0
#    （即「增强不如原始 200 条本身」）。这本身就是一个诚实的结果，
#    但为了演示「决策随标注单价翻转」，这里沿用第 6 节那个保守示意值。
if not math.isfinite(n_eq200) or n_eq200 <= 200:
    print(f'（实测 n_eq={n_eq200:.0f} <= 200 -> 该点增强无净收益；改用示意值 350 演示决策）')
    n_eq200 = 350.0
line = report_one_liner(v_f, n_eq200, 200, 0.5, 480.0)
print(line)
kv = dict(p.split('=') for p in line.split())
assert set(kv) == {'delta', 'p', 'vs_dup', 'n_eq', 'value', 'cost', 'decision'}
assert kv['decision'] in ('DO', 'SKIP')
# 标注很贵时决策应翻转
line_med = report_one_liner(v_f, n_eq200, 200, 20.0, 480.0)
kv_med = dict(p.split('=') for p in line_med.split())
print(line_med)
assert kv_med['decision'] == 'DO', '专家标注领域应判 DO'
print('\n✅ 胶囊练习通过：一行摘要接进 C37 的实验追踪，就有了增强决策的完整记录。')
print('   注意同一个技术结果，在两个领域给出了相反的决策 —— **这才是正确的决策方式**。')

In [ ]:
# 📖 胶囊参考答案
def report_one_liner(verdict_dict, n_eq, n0, cost_per_label, aug_cost):
    ca, cb = verdict_dict['C_vs_A'], verdict_dict['C_vs_B']
    saved = max(0.0, n_eq - n0) if math.isfinite(n_eq) else 0.0
    value = saved * cost_per_label
    dec = 'DO' if value > aug_cost else 'SKIP'
    return (f'delta={ca["mean"]:+.4f} p={ca["p_le_zero"]:.4f} '
            f'vs_dup={cb["mean"]:+.4f} n_eq={n_eq:.0f} '
            f'value=${value:.0f} cost=${aug_cost:.0f} decision={dec}')

### 小结
- 增强文献的可信度危机由三点叠加：**效应量小 + 噪声大 + 增强天然带混淆**（样本数、训练步数、正则化强度、lr 调度全都变了）。
- **必须固定优化步数而不是 epoch 数**——已可运行地演示「不固定时 Δ 被高估」。
- **四个对照组**：A baseline / **B 复制对照** / C 增强 / **D 真实数据对照**。
  **C > B 才说明多样性有贡献**；**C > D 才说明这笔预算花在增强上更值**。只报 C vs A 无法排除任何混淆。
- **配对是免费的方差削减**（同种子高度正相关，差值方差显著更小）→ 同样种子数能检出更小效应。
- **配对 bootstrap** 不假设正态；先做 **A/A 检验**验证方法本身。
- **功效分析要在跑实验之前做**：若「要 30 个种子」而你只跑 3 个，这个实验从设计上无法得出结论。
- **学习曲线 > 单点比较**；从它读出**等价真实数据量**——「增强 4× ≈ 多 150 条真实数据」比「涨了 1.2 分」有用一百倍。
- **三条决策规律**：①标注成本决定一切（同样技术收益在通用文本不划算、在医疗法律非常划算）；②数据量越大越不划算；③**零标注冷启动是增强不可替代的场景**（该问「多快上线」而非「划不划算」）。

🎓 **本课完结。** 你现在能造数据（模块 01–03）、筛数据（模块 04）、并**证明或推翻它有用**（模块 05）。
最重要的是那个换算：**把增强的收益变成「等价真实数据量」，一切就都可比了。**
建议的下一站：**C10**（测量科学与 A/B）、**C03**（评测统计）、**C43**（大规模数据工程）、**C40**（研究方法论）。